In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import spikeinterface as si
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm
import spikeinterface.exporters as sexp


In [2]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup
from probeinterface.plotting import plot_probe, plot_probegroup
from probeinterface import generate_dummy_probe, generate_linear_probe
from probeinterface import write_probeinterface, read_probeinterface
from probeinterface import write_prb, read_prb

In [3]:
date_list = os.listdir("/media/ubuntu/sda/data/mouse6/ns4/natural_image/")
default_params = {
        'detect_sign': -1,  # Use -1, 0, or 1, depending on the sign of the spikes in the recording
        'adjacency_radius': 120,  # Use -1 to include all channels in every neighborhood
        'freq_min': 300,  # Use None for no bandpass filtering
        'freq_max': 3000,
        'filter': True,
        'whiten': True,  # Whether to do channel whitening as part of preprocessing
        'num_workers': 9,
        'clip_size': 50,
        'detect_threshold': 4, # 5
        'detect_interval': 3,  # Minimum number of timepoints between events detected on the same channel, 30
    }
for date in date_list:
    date = date.split("_")[1]
    recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/mouse6_{date}_natural_image_001.ns4')
    recording_recorded = recording_raw.remove_channels(['31', '32', '98'])
    probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
    recording_recorded = recording_recorded.set_probegroup(probe_30channel)
    recording_cmr = recording_recorded
    recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
    recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
    recording_preprocessed = recording_cmr.save(format="binary")

    output_folder = f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_mountainsort/{date}"
    os.makedirs(output_folder, exist_ok=True)
    sorting_wave_clus = ss.run_sorter(sorter_name='mountainsort4',
                                     recording=recording_preprocessed,
                                     remove_existing_folder='True',
                                     folder=output_folder,
                                     **default_params,)
    analyzer_mountainsort4 = si.create_sorting_analyzer(
        sorting=sorting_wave_clus, 
        recording=recording_f, 
        format='binary_folder', 
        folder=output_folder + '/analyzer'
    )

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_mountainsort4.compute(extensions_to_compute, extension_params=extension_params)

    qm_params = sqm.get_default_qm_params()
    analyzer_mountainsort4.compute("quality_metrics", qm_params)

    sexp.export_to_phy(analyzer_mountainsort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

Use cache_folder=/tmp/spikeinterface_cache/tmp7mpuvp3k/50SRACPY
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=585.94 KiB - chunk_duration=1.00s


write_binary_recording (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.


estimate_sparsity (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

...

spike_amplitudes (no parallelization):   0%|          | 0/2540 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/52 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/52 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/2540 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_mountainsort/122022/phy_folder_for_kilosort/params.py
